In [ ]:
repository_filter: list[str] = []
top_n_methods: str = "8"

In [ ]:
from code_data_science import data_table as dt
from moderne_visualizations_misc.reusable import quality_utils as qu
import plotly.graph_objects as go
import numpy as np
import warnings

warnings.simplefilter("ignore")

top_n = int(top_n_methods)

df = dt.read_csv("../samples/method_quality_metrics.csv")
df = qu.filter_repos(df, repository_filter)
df = qu.add_repo_short(df)
df = qu.add_class_short(df)

In [ ]:
radar_metrics = [
    "cyclomaticComplexity",
    "cognitiveComplexity",
    "maxNestingDepth",
    "parameterCount",
    "halsteadEstimatedBugs",
]
radar_labels = ["Cyclomatic", "Cognitive", "Nesting", "Parameters", "Est. Bugs"]

# Check required columns exist
missing = [c for c in radar_metrics if c not in df.columns]
if len(df) == 0 or missing:
    msg = "No data available." if len(df) == 0 else f"Missing columns: {missing}"
    fig = qu.empty_figure(msg)
else:
    # Normalize each metric to 0-1
    norm_df = df[radar_metrics].copy()
    for col in radar_metrics:
        col_max = norm_df[col].max()
        if col_max > 0:
            norm_df[col] = norm_df[col] / col_max
        else:
            norm_df[col] = 0

    # Rank by sum of normalized metrics
    norm_df["risk_sum"] = norm_df[radar_metrics].sum(axis=1)
    top_indices = norm_df.nlargest(min(top_n, len(norm_df)), "risk_sum").index

    fig = go.Figure()
    colors = [
        "#EF5350", "#FF8A65", "#FFB74D", "#FFE082",
        "#A5D6A7", "#4DB6AC", "#4FC3F7", "#7986CB",
        "#BA68C8", "#F06292",
    ]

    for rank, idx in enumerate(top_indices):
        row = norm_df.loc[idx]
        orig = df.loc[idx]
        values = [row[m] for m in radar_metrics]
        # Close the polygon
        values_closed = values + [values[0]]
        labels_closed = radar_labels + [radar_labels[0]]

        method_label = f"{orig.get('classShort', '')}::{orig['methodName']}"
        color = colors[rank % len(colors)]

        fig.add_trace(
            go.Scatterpolar(
                r=values_closed,
                theta=labels_closed,
                fill="toself",
                fillcolor=color.replace(")", ",0.15)").replace("#", "rgba("),
                opacity=0.8,
                name=method_label[:50],
                line=dict(color=color, width=2),
                customdata=[
                    [
                        orig["cyclomaticComplexity"],
                        orig["cognitiveComplexity"],
                        orig["maxNestingDepth"],
                        orig["parameterCount"],
                        orig["halsteadEstimatedBugs"],
                    ]
                ] * len(labels_closed),
                hovertemplate=(
                    f"<b>{method_label[:40]}</b><br>"
                    "Cyclomatic: %{customdata[0]}<br>"
                    "Cognitive: %{customdata[1]}<br>"
                    "Nesting: %{customdata[2]}<br>"
                    "Params: %{customdata[3]}<br>"
                    "Est Bugs: %{customdata[4]:.2f}"
                    "<extra></extra>"
                ),
            )
        )

    fig.update_layout(
        polar=dict(
            radialaxis=dict(visible=True, range=[0, 1.05]),
        ),
        title=f"Method Risk Radar (top {min(top_n, len(top_indices))} riskiest methods)",
        height=700,
        width=800,
        margin=dict(l=80, r=80, t=80, b=60),
        legend=dict(
            font=dict(size=10),
            orientation="v",
            x=1.1, y=1,
        ),
    )
fig.show()